# Load API Key and Libraries

In [ ]:
# Load Legiscan API Key from Notebook Secrets
from google.colab import userdata
API_KEY = userdata.get('legiscan')

In [ ]:
# Import packages

import os
import requests
import zipfile
import io
import json
import pandas as pd
from tqdm.notebook import tqdm
import time
from pprint import pprint

# Find, fetch, and process JSON dataset

In [ ]:
def get_dataset_list():
    """
    Get list of available datasets from LegiScan API
    """
    url = f"https://api.legiscan.com/?key={API_KEY}&op=getDatasetList"
    response = requests.get(url)
    if response.status_code == 200:
        print(f'Success!')
        data = response.json()
        return data

    else:
        print(f"Error fetching dataset list: {response.status_code}")
        return None

In [ ]:
print(f'Fetching the available datasets from Legiscan...')
datasets = get_dataset_list()

Fetching the available datasets from Legiscan...
Success!


In [ ]:
# Find the relevant dataset based on the search keyword
def dataset_search(keyword, datasets):
    '''

    '''
    count = 0
    for data in datasets['datasetlist']:
        if search_keyword in data['session_name']:
            count += 1
            target = data
        else:
            continue
    if target == None:
            print('Keyword not found!')
    elif count > 1:
        raise ValueError(f'{count} datasets found! Try again with a more narrow search.')
    else:
        print(f'Success! {count} dataset found.')
        return target


In [ ]:
# Find dataset for the 118th Congress
search_keyword = '118th Congress'

target = dataset_search(search_keyword, datasets)

Success! 1 dataset found.


In [ ]:
# Pull identifying information for target dataset (for data loading)
session_id = target['session_id']
access_key = target['access_key']


print(f'The session key for the 118th congress is: {session_id}')
print(f'The access key for the 118th congress is: {access_key}')

The session key for the 118th congress is: 2041
The access key for the 118th congress is: 1C0tmRQkwY2bmu3vz8cXix


In [ ]:
def get_specific_dataset(session_id, access_key):
    data_url = f'https://api.legiscan.com/?key={API_KEY}&op=getDataset&id={session_id}&access_key={access_key}'
    response = requests.get(data_url)
    if response.status_code == 200:
        data = response.json()
        return data
    else:
        print(f"Error fetching dataset list: {response.status_code}")

In [ ]:
data = get_specific_dataset(session_id, access_key)

In [ ]:
print(f'They keys to the dataset are:\n ------')
for key in data['dataset'].keys():
    print(key)

encoded_zip = data['dataset']['zip']


print(f'\nThe encoded dataset zipfile information is in a {type(encoded_zip)} format.')
print(f'This encoding follows the Base64 data encoding protocol.')


They keys to the dataset are:
 ------
state_id
session_id
year_start
year_end
prefile
sine_die
prior
special
session_tag
session_title
session_name
dataset_date
dataset_hash
dataset_size
dataset_size_csv
mime_type
zip

The encoded dataset zipfile information is in a <class 'str'> format.
This encoding follows the Base64 data encoding protocol.


In [ ]:
import base64
import zipfile
import io

# Following code created with assistance from Claude 3.7

# Correct padding if necessary
def correct_base64_padding(encoded_str):
    '''
    corrects padding of base64 encoding string
    Args:
        input:
            encoded string (str)
        output:
            encoded string (str)

    '''
    missing_padding = len(encoded_str) % 4
    if missing_padding:
        encoded_str += '=' * (4 - missing_padding)
    return encoded_str

def load_encoded_dataset(encoded_zip, filename):
    '''loads encoded_zip file into specified directory
    Args:
        input:
            encoded string (str)
            directory filename (str)
        output:
            directory path (str)
    '''

    if os.path.exists(filename):
        print(f'The file {filename} already exists. Create a new directory filename and try again.')
        return
    try:
        # Correct padding
        encoded_zip = correct_base64_padding(encoded_zip)

        # Decode the Base64 string
        decoded_zip = base64.b64decode(encoded_zip)

        # Create a BytesIO object from the decoded data
        zip_file = io.BytesIO(decoded_zip)

        # Open and extract the ZIP file
        with zipfile.ZipFile(zip_file, 'r') as zf:
            # Extract all files to a directory
            zf.extractall(filename)
            # Save the absolute path of the extraction directory
        extracted_path = os.path.abspath(filename)
        print(f"Files extracted to: {extracted_path}")
    # if it doesn't work, raise the error messages
    except base64.binascii.Error as e:
        print("Base64 decoding error:", e)
    except zipfile.BadZipFile as e:
        print("Invalid ZIP file:", e)
    except Exception as e:
        print("Error processing ZIP file:", e)

    return extracted_path

In [ ]:
# Load encoded dataset; save in directory of the given filename
directory_path = load_encoded_dataset(encoded_zip, search_keyword)

Files extracted to: /content/118th Congress


In [ ]:
import os
import json

# Create directory path to bills containing JSON files
bills_path = directory_path + '/US/2023-2024_118th_Congress/bill'

# Create an empty dictionary to store the results
bills_initial = {}

# Loop through all JSON files in the directory
for filename in os.listdir(bills_path): # for file in the folder
    if filename.endswith('.json'): # if it is a .json file
        file_path = os.path.join(bills_path, filename) # create the file path
        try:
            # Open and read the JSON file
            with open(file_path, 'r', encoding='utf-8') as file:
                # Parse JSON content and add to dictionary with filename as key
                bills_initial[filename] = json.load(file)

            # Print progress to user
            if len(bills_initial) % 1000 == 0:
                print(f"Processed {len(bills_initial):,} files...")

        except json.JSONDecodeError as e:
            print(f"Error parsing {filename}: {e}")
        except Exception as e:
            print(f"Error processing {filename}: {e}")

print(f"Loaded {len(bills_initial):,} JSON files into a dictionary")


Processed 1,000 files...
Processed 2,000 files...
Processed 3,000 files...
Processed 4,000 files...
Processed 5,000 files...
Processed 6,000 files...
Processed 7,000 files...
Processed 8,000 files...
Processed 9,000 files...
Processed 10,000 files...
Processed 11,000 files...
Processed 12,000 files...
Processed 13,000 files...
Processed 14,000 files...
Processed 15,000 files...
Processed 16,000 files...
Processed 17,000 files...
Processed 18,000 files...
Processed 19,000 files...
Loaded 19,304 JSON files into a dictionary


# Filter bills

In [ ]:
def filter_bills(bill_input, bill_number = None, body = None):
    '''filters bills based on selections, returns a dictionary of bills
    Args:
        input:
            bill_input: dictionary of bills (bill number: bill details)
            bill_number: specified bill number (e.g., HR2500) (str)
            body: letter identifier for Chamber of Congress (e.g., H) (str)
        output:
            bills: dictionary of filtered bills (bill number: bill details)
    '''
    # Initialize empty dictionary to store filtered bills
    bills = {}

    # Iterate through given bills
    for bill in bill_input:

        # filter on bill input
        if bill_input: # if criteria is not NoneType
            if bill_input[bill]['bill']['body'] != body: # if value does not match input
                continue # continue to next iteration

        # filter on bill number
        if bill_number: # if criteria is not NoneType
            bill_number = bill_number.upper() # convert to uppercase
            if bill_input[bill]['bill']['bill_number'].upper() != bill_number: # if value does not match input
                continue # continue to next iteration

        # add bill to dictionary
        bills[bill] = bill_input[bill]
    return bills

In [ ]:
filtered_bills = filter_bills(bills_initial, body = 'H')
# filtered_bills = filter_bills(bills_initial, body = 'H', bill_number = 'hb3091')

In [ ]:
# create list of bill keys returned
filtered_bill_keys = list(filtered_bills.keys())

print(f'The number of bills found is: {len(filtered_bill_keys):,} bills')

The number of bills found is: 12,545 bills


# Import bill action milestone (Reported to House)



In [ ]:
from google.colab import drive

# connect to Drive
drive.mount('/content/gdrive/', force_remount=True)

# define file path
milestone_path = 'milestones.csv'

# import data
milestone_data = pd.read_csv(milestone_path, skiprows=3)

Mounted at /content/gdrive/


In [ ]:
milestone_data.columns

Index(['Legislation Number', 'URL', 'Congress', 'Title', 'Sponsor',
       'Date of Introduction', 'Committees', 'Latest Action',
       'Latest Action Date', 'Number of Cosponsors', 'Amends Bill',
       'Date Offered', 'Date Submitted', 'Date Proposed', 'Amends Amendment'],
      dtype='object')

In [ ]:
# parse bill type and bill number
# use regex

# Remove spaces and periods
milestone_data['cleaned_bill'] = milestone_data['Legislation Number'].str.replace('[ .]', '', regex=True)

# Split into text and numeric components
milestone_data[['bill_type', 'bill_num']] = milestone_data['cleaned_bill'].str.extract(
    r'^([A-Za-z]+)(\d+)$',  # Regex pattern
    expand=True
)


# Create bill attribute dataframe

In [ ]:
import statistics
from datetime import datetime
from collections import Counter
import pandas as pd


def get_bill_attributes(bill):
    '''
    Parses JSON bill structure for desired attributes
    Args:
        input: bill (dict)
        output: attributes (dict)
    '''
    # Bill title
    try:
        bill_title = bill['bill']['title']
    except:
        bill_title = None

    # Bill number
    try:
        bill_number = bill['bill']['bill_number']
    except:
        bill_number = None

    # Body
    try:
        body = bill['bill']['body']
    except:
        body = None

    # Bill type
    try:
        bill_type = bill['bill']['bill_type']
    except:
        bill_type = None

    # Status
    try:
        status = bill['bill']['status']
    except:
        status = None

    # Sponsor party
    try:
        sponsor_party = bill['bill']['sponsors'][0]['party']
    except:
        sponsor_party = None

    # Sponsor ID
    try:
        sponsor_people_id = bill['bill']['sponsors'][0]['people_id']
    except:
        sponsor_people_id = None

    # Sponsor first name
    try:
        sponsor_first_name = bill['bill']['sponsors'][0]['first_name']
    except:
        sponsor_first_name = None

    # Sponsor last name
    try:
        sponsor_last_name = bill['bill']['sponsors'][0]['last_name']
    except:
        sponsor_last_name = None

    # Sponsor hash
    try:
        sponsor_hash = bill['bill']['sponsors'][0]['person_hash']
    except:
        sponsor_hash = None

    # Sponsor state
    try:
        sponsor_state = bill['bill']['sponsors'][0]['district'].split("-")[1]
    except:
        sponsor_state = None

    # Number of cosponsors
    try:
        num_cosponsors = len(bill['bill']['sponsors']) - 1  # Subtract 1 to exclude the main sponsor
    except:
        num_cosponsors = 0

    # Cosponsor parties (list)
    try:
        cosponsor_parties = [sponsor['party'] for sponsor in bill['bill']['sponsors'][1:]]
    except:
        cosponsor_parties = []

    # Cosponsor states (list)
    try:
        cosponsor_states = [sponsor['district'].split("-")[1] for sponsor in bill['bill']['sponsors'][1:]]
    except:
        cosponsor_states = []

    # Number of cosponsor states
    try:
        num_cosponsor_states = len(set(cosponsor_states)) # set to keep only unique values
    except:
        num_cosponsor_states = None

    # % cosponsor Democrat party
    try:
        per_cosponsor_parties_D = (cosponsor_parties.count('D') / len(cosponsor_parties))*100
    except:
        per_cosponsor_parties_D = None

    if per_cosponsor_parties_D == None:
        per_cosponsor_parties_D = 0

    # % cosponsor Republican
    try:
        per_cosponsor_parties_R = (cosponsor_parties.count('R') / len(cosponsor_parties))*100
    except:
        per_cosponsor_parties_R = None

    if per_cosponsor_parties_R == None:
        per_cosponsor_parties_R = 0

    # Committees and sub-committees
    committees = []
    subcommittees = []

    try:
        for referral in bill['bill']['referrals']:
            name = referral['name']
            chamber = referral['chamber']
            if 'Subcommittee' in name:
                if chamber == 'H':
                    subcommittees.append(name)
            elif chamber == 'H':
                    committees.append(name)
    except:
        pass

    # Number of committees
    num_committees = len(set(committees))

    # Number of sub-committees
    num_subcommittees = len(set(subcommittees))

    # Subject area
    try:
        subject_areas = [subject['subject_name'] for subject in bill['bill']['subjects']]
    except:
        subject_areas = []

    # Number of subject areas
    num_subject_areas = len(set(subject_areas))

    # Number of related bills
    try:
        num_related_bills = len(bill['bill']['sasts'])
    except:
        num_related_bills = None


    # Time to introduction / union calendar
    try:
        introduction_date = None
        union_calendar_date = None

        # Find introduction date
        for event in bill['bill']['history']:
            if "Introduced" in event['action']:
                introduction_date = event['date']
                break

        # Find Union Calendar date
        for event in bill['bill']['history']:
            if "Calendar" in event['action']:
                union_calendar_date = event['date']
                break

        # Calculate days to Union Calendar if both dates exist
        if introduction_date and union_calendar_date:
            from datetime import datetime
            intro_date_obj = datetime.strptime(introduction_date, '%Y-%m-%d')
            calendar_date_obj = datetime.strptime(union_calendar_date, '%Y-%m-%d')
            days_to_union_calendar = (calendar_date_obj - intro_date_obj).days
        else:
            days_to_union_calendar = None
    except:
        introduction_date = None
        union_calendar_date = None
        days_to_union_calendar = None

    # Time from introduction to first action
    try:
        # Skip introduction and find first action
        first_action_date = None
        first_action_description = None

        # Find the introduction event first
        introduction_found = False
        for event in bill['bill']['history']:
            if "Introduced" in event['action']:
                introduction_found = True
                continue

            # Find event after the introduction
            if introduction_found:
                first_action_date = event['date']
                first_action_description = event['action']
                break

        # Calculate days from introduction to first action
        if introduction_date and first_action_date:
            intro_date_obj = datetime.strptime(introduction_date, '%Y-%m-%d')
            action_date_obj = datetime.strptime(first_action_date, '%Y-%m-%d')
            days_to_first_action = (action_date_obj - intro_date_obj).days
        else:
            days_to_first_action = None
    except:
        first_action_date = None
        first_action_description = None
        days_to_first_action = None

    # Bill complexity (using text file size as proxy)
    try:
        bill_texts = bill['bill']['texts']
        latest_text = max(bill_texts, key=lambda x: x['date'])
        text_file_size = latest_text['text_size']

    except:
        text_file_size = None

    attributes = {
        'title': bill_title,
        'bill_number': bill_number,
        'body': body,
        'bill_type': bill_type,
        'status': status,
        'sponsor_party': sponsor_party,
        'sponsor_people_id': sponsor_people_id,
        'sponsor_hash': sponsor_hash,
        'sponsor_first_name': sponsor_first_name,
        'sponsor_last_name': sponsor_last_name,
        'sponsor_state': sponsor_state,
        'num_cosponsors': num_cosponsors,
        'cosponsor_parties': cosponsor_parties,
        'cosponsor_states': cosponsor_states,
        'num_cosponsor_states': num_cosponsor_states,
        'per_cosponsor_parties_D': per_cosponsor_parties_D,
        'per_cosponsor_parties_R': per_cosponsor_parties_R,
        'committees_house': committees,
        'sub-committees_house': subcommittees,
        'num_committees_house': num_committees,
        'num_subcommittees_house': num_subcommittees,
        'subject_areas': subject_areas,
        'num_subject_areas': num_subject_areas,
        'num_related_bills': num_related_bills,
        'introduction_date': introduction_date,
        'text_file_size': text_file_size
    }

    return attributes


In [ ]:
bill_data_parsed = []

for bill in filtered_bills:
    data = get_bill_attributes(filtered_bills[bill])
    bill_data_parsed.append(data)

In [ ]:
# Create dataframe
bills_data_parsed_df = pd.DataFrame(bill_data_parsed)
bills_data_parsed_df.iloc[1:5]

,title,bill_number,body,bill_type,status,sponsor_party,sponsor_people_id,sponsor_hash,sponsor_first_name,sponsor_last_name,...,per_cosponsor_parties_R,committees_house,sub-committees_house,num_committees_house,num_subcommittees_house,subject_areas,num_subject_areas,num_related_bills,introduction_date,text_file_size
1,EEOC Transparency and Accountability Act,HB1054,H,B,1,R,11077,t1xxilhf,Tim,Walberg,...,0.0,[Education And The Workforce],[],1,0,"[Civil rights and liberties, minority issues]",1,1,2023-02-14,231872.0
2,Natural Disaster Recovery Program Act of 2024,HB9750,H,B,1,R,16476,fds8xyci,David,Rouzer,...,100.0,[Transportation And Infrastructure],"[Subcommittee on Economic Development, Public ...",1,1,[],0,1,2024-09-23,270704.0
3,Year-Round E15 Act of 2024,HB8052,H,B,1,R,24053,d6p4ilmt,Zachary,Nunn,...,60.0,[Energy And Commerce],"[Subcommittee on Environment, Manufacturing, a...",1,1,[Environmental protection],1,0,2024-04-17,231004.0
4,Health Equity and MENA Community Inclusion Act...,HB2730,H,B,1,D,20115,pie8gfo6,Rashida,Tlaib,...,0.0,[Energy And Commerce],[],1,0,[Health],1,1,2023-04-19,241502.0


In [ ]:
# Write to a .csv

# Convert column of lists into strings

csv_export_df = bills_data_parsed_df
csv_export_df['cosponsor_parties'] = bills_data_parsed_df['cosponsor_parties'].apply(str)
csv_export_df['sub-committees_house'] = bills_data_parsed_df['sub-committees_house'].apply(str)
csv_export_df['subject_areas'] = bills_data_parsed_df['subject_areas'].apply(str)

# Merge bill identifiers between sets

In [ ]:
# milestone data identifiers

# Define Milestone data ordered categories
category_order = list(milestone_data['bill_type'].unique())
print(f'Milestone (Congress.gov) identifiers: {category_order}')
target_labels = ['HR', 'HJRes', 'HConRes', 'HRes']
print(f'Target labels: {target_labels}')

# Convert to categorical type
milestone_data['bill_type_mapped'] = pd.Categorical(
    milestone_data['bill_type'],
    categories=category_order,
    ordered=True
).rename_categories(target_labels)

print('------')

# bill data identifiers

# Define Milestone data ordered categories
category_order = list(bills_data_parsed_df['bill_type'].unique())
print(f'Bills (Legiscan) identifiers: {category_order}')
target_labels = ['HR', 'HRes', 'HJRes', 'HConRes']
print(f'Target labels: {target_labels}')

# Convert to categorical type
bills_data_parsed_df['bill_type_mapped'] = pd.Categorical(
    bills_data_parsed_df['bill_type'],
    categories=category_order,
    ordered=True
).rename_categories(target_labels)

Milestone (Congress.gov) identifiers: ['HR', 'HJRes', 'HConRes', 'HRes']
Target labels: ['HR', 'HJRes', 'HConRes', 'HRes']
------
Bills (Legiscan) identifiers: ['B', 'R', 'JR', 'CR']
Target labels: ['HR', 'HRes', 'HJRes', 'HConRes']


In [ ]:
milestone_data['bill_type_mapped'].unique()

['HR', 'HJRes', 'HConRes', 'HRes']
Categories (4, object): ['HR' < 'HJRes' < 'HConRes' < 'HRes']

In [ ]:
bills_data_parsed_df['bill_type_mapped'].unique()

['HR', 'HRes', 'HJRes', 'HConRes']
Categories (4, object): ['HR' < 'HRes' < 'HJRes' < 'HConRes']

In [ ]:
# Get bill number isolated for parsed data

bills_data_parsed_df['bill_number_clean'] = bills_data_parsed_df['bill_number'].str.extract(r'(\d+)')

In [ ]:
# concatenate recoded bill type and number

milestone_data['bill_id_concat'] = milestone_data['bill_type_mapped'].astype(str) + milestone_data['bill_num'].astype(str)
bills_data_parsed_df['bill_id_concat'] = bills_data_parsed_df['bill_type_mapped'].astype(str) + bills_data_parsed_df['bill_number_clean'].astype(str)

In [ ]:
print(f'The Congress.Gov set has {len(milestone_data["bill_id_concat"].unique()):,} unique records.')

print(f'The Legiscan set has {len(bills_data_parsed_df["bill_id_concat"].unique()):,} unique records')

The Congress.Gov set has 940 unique records.
The Legiscan set has 12,545 unique records


In [ ]:
# Tag response in main parsed dataframe

missing_matches = [
    bill_id
    for bill_id in milestone_data['bill_id_concat']
    if bill_id not in bills_data_parsed_df['bill_id_concat'].values
]

if(len(missing_matches) > 0):
    raise Warning('Dataset match not perfect')
else:
    print('Dataset match perfect!')

Dataset match perfect!


In [ ]:
# Complete data match, create target column

bills_data_parsed_df['reported_to_floor'] = bills_data_parsed_df['bill_id_concat'].isin(milestone_data['bill_id_concat'])

# States and Codes

In [ ]:
import requests
from bs4 import BeautifulSoup

In [ ]:
# pull reference for state / abbreviations
# census website was forbidden to web s
url = "https://www.faa.gov/air_traffic/publications/atpubs/cnt_html/appendix_a.html"

request = requests.get(url)

request.status_code

soup = BeautifulSoup(request.text)

In [ ]:
# Initialize an empty dictionary to hold state: abbreviation pairs
state_dict = {}

# Find all rows in the table
rows = soup.find_all("tr")

# Iterate through each row and extract table cells
for row in rows:
    cells = row.find_all("td")
    # Process cells in pairs
    for i in range(0, len(cells), 2):
        if i + 1 < len(cells):
            state = cells[i].get_text(strip=True)
            abbreviation = cells[i+1].get_text(strip=True)
            state_dict[state] = abbreviation

In [ ]:
state_dict_df = pd.DataFrame(list(state_dict.items()), columns=['state', 'abbreviation'])
state_dict_df.head()

,state,abbreviation
0,Alabama,AL
1,Kentucky,KY
2,Ohio,OH
3,Alaska,AK
4,Louisiana,LA


# Gather Congressional Representiave data

In [ ]:
# Import API Key

from google.colab import userdata

api_key_congress = userdata.get('CONGRESS.GOV_API')

In [ ]:
import requests
from pprint import pprint
import pandas as pd

In [ ]:
def load_members(congress_number):

    ''' load all members from a given Congress
        Args:
            congress_number (int)
        @returns: TBD
    '''

    # define URL for API request
    request_url = f'https://api.congress.gov/v3/member/congress/{congress_number}?limit=250&api_key={api_key_congress}'

    # try to request data
    try:
        response = requests.get(request_url)
        response.raise_for_status()
    # if request has an error, return the error
    except requests.exceptions.HTTPError as err:
        print(err)

    # parse data
    data = response.json()

    # create list of members
    members = data['members']

    # determine number of total members from first data response
    limit = data['pagination']['count']

    # print message on total bills to fetch
    print(f'Fetching {limit:,} members...', '\n', '----------')

    # continue searching if the number of members is less than the total
    while len(members) < limit:
        # provide user a progress update
        print(f'Fetched {len(members)/limit:.0%} so far...')
        # get next URL
        request_url = f"{data['pagination']['next']}&api_key={api_key_congress}"
        # try to fetch data
        try:
            response = requests.get(request_url)
            response.raise_for_status()
        # if request has an error, return the error
        except requests.exceptions.HTTPError as err:
            print(err)
        # parse data
        data = response.json()
        # add members to existing list
        members.extend(data['members'])
    # return list of members
    print('Done!')
    return members


In [ ]:
members = load_members(118)

Fetching 554 members... 
 ----------
Fetched 45% so far...
Fetched 90% so far...
Done!


In [ ]:
members[0]

{'bioguideId': 'S001208',
 'depiction': {'attribution': 'Image courtesy of the Member',
  'imageUrl': 'https://www.congress.gov/img/member/s001208_200.jpg'},
 'district': 7,
 'name': 'Slotkin, Elissa',
 'partyName': 'Democratic',
 'state': 'Michigan',
 'terms': {'item': [{'chamber': 'House of Representatives',
    'endYear': 2025,
    'startYear': 2019},
   {'chamber': 'Senate', 'startYear': 2025}]},
 'updateDate': '2025-03-14T12:42:43Z',
 'url': 'https://api.congress.gov/v3/member/S001208?format=json'}

In [ ]:
def parse_member(member):
    '''
    DEFINE

    '''
    # record most recent chamber
    try:
        recent_chamber = member['terms']['item'][-1]['chamber']
    except:
        recent_chamber = None
    # record name
    try:
        name = member['name']
    except:
        name = None
    # record startYear
    try:
        start_year = member['terms']['item'][-1]['startYear']
    except:
        start_year = None

    # record last name
    try:
        last_name = member['name'].split(',')[0]
    except:
        last_name = None

    # record first name
    try:
        first_name = member['name'].split(',')[-1].split(' ')[1]
    except:
        first_name = None

    # record state
    try:
        state = member['state']
    except:
        state = None

    attributes = {
        'recent_chamber': recent_chamber,
        'name': name,
        'start_year': start_year,
        'last_name': last_name,
        'first_name': first_name,
        'state': state
    }
    return attributes

In [ ]:
member_data_all = []

for member in members:
    member_data_all.append(parse_member(member))

In [ ]:
# Option to filter for members of house
member_data = [member for member in member_data_all if "" in member['recent_chamber']]

In [ ]:
member_data_df = pd.DataFrame(member_data)
print(member_data_df.shape)
member_data_df.head()

(554, 6)


,recent_chamber,name,start_year,last_name,first_name,state
0,Senate,"Slotkin, Elissa",2025,Slotkin,Elissa,Michigan
1,House of Representatives,"Grijalva, Raúl M.",2003,Grijalva,Raúl,Arizona
2,Senate,"Banks, Jim",2025,Banks,Jim,Indiana
3,Senate,"Young, Todd",2017,Young,Todd,Indiana
4,Senate,"Wyden, Ron",1996,Wyden,Ron,Oregon


In [ ]:
# Join state abbreviation to state names
# left merge to keep all members and join states

member_data_parsed_df = pd.merge(member_data_df, state_dict_df, on = 'state', how = 'left')

# create merged field (last name, first name initial, state abbreviation)
member_data_parsed_df['name_state'] = member_data_parsed_df['last_name'] + member_data_parsed_df['first_name'].str[0] + member_data_parsed_df['abbreviation']

In [ ]:
member_data_parsed_df.head()

,recent_chamber,name,start_year,last_name,first_name,state,abbreviation,name_state
0,Senate,"Slotkin, Elissa",2025,Slotkin,Elissa,Michigan,MI,SlotkinEMI
1,House of Representatives,"Grijalva, Raúl M.",2003,Grijalva,Raúl,Arizona,AZ,GrijalvaRAZ
2,Senate,"Banks, Jim",2025,Banks,Jim,Indiana,IN,BanksJIN
3,Senate,"Young, Todd",2017,Young,Todd,Indiana,IN,YoungTIN
4,Senate,"Wyden, Ron",1996,Wyden,Ron,Oregon,OR,WydenROR


In [ ]:
# check if identier is unique
member_data_parsed_df['name_state'].value_counts()

member_data_parsed_df[member_data_parsed_df.duplicated(subset=['name_state'], keep=False)]

,recent_chamber,name,start_year,last_name,first_name,state,abbreviation,name_state
99,Senate,"Menendez, Robert",2006,Menendez,Robert,New Jersey,NJ,MenendezRNJ
138,House of Representatives,"Brown, Shontel M.",2021,Brown,Shontel,Ohio,OH,BrownSOH
149,House of Representatives,"Menendez, Robert",2023,Menendez,Robert,New Jersey,NJ,MenendezRNJ
300,Senate,"Brown, Sherrod",2007,Brown,Sherrod,Ohio,OH,BrownSOH


# Gather Congress Data

In [ ]:
def load_congress(congress_number):

    ''' load details for a given Congress
        @params: congress_number (int)
        @returns:

    '''
    # define URL for API request
    request_url = f'https://api.congress.gov/v3/congress/{congress_number}?api_key={api_key_congress}'
    # try to request data
    try:
        response = requests.get(request_url)
        response.raise_for_status()
    # if request has an error, return the error
    except requests.exceptions.HTTPError as err:
        print(err)

    # parse data
    data = response.json()

    return data

In [ ]:
congress_number = 118
congress_data = load_congress(congress_number)

In [ ]:
# congress_data

In [ ]:
# Determine start date of the Congress
from datetime import datetime

start_dates = []
end_dates = []
for item in congress_data['congress']['sessions']:
    date = (item['startDate'])
    date_end = (item['endDate'])
    start_dates.append(date)
    end_dates.append(date_end)

# find earliest start date
congress_start_date = min(start_dates)

# find last end date
congress_end_date = max(end_dates)

# convert to datetime objects
congress_start_date_obj = datetime.strptime(congress_start_date, '%Y-%m-%d')
congress_end_date_obj = datetime.strptime(congress_end_date, '%Y-%m-%d')
print(f'Start: {congress_start_date_obj}')
print(f'End: {congress_end_date_obj}')

Start: 2023-01-03 00:00:00
End: 2025-01-03 00:00:00


In [ ]:
# Determine control of the House
parties = [item['type'] for item in congress_data['congress']['sessions']
           if item['chamber'] == 'House of Representatives']

# if party values are the same, return value
if len(set([x.upper() for x in parties])) == 1:
    house_party_control = parties[0]
else:
    raise ValueError('Party control changed during Congress session.')

print(f'The {house_party_control} party had control of the House during the {congress_number} Congress. ')

The R party had control of the House during the 118 Congress. 


In [ ]:
# get end year of 118th Congress

congress_end_year = congress_end_date_obj.year
print(congress_end_year)

2025


In [ ]:
# calculate most recent tenure Congress person length

for index in member_data_parsed_df.index:  # Iterate through each row index
    member_data_parsed_df.loc[index, 'current_tenure_len'] = congress_end_year - member_data_parsed_df.loc[index, 'start_year']

In [ ]:
member_data_parsed_df.head()

,recent_chamber,name,start_year,last_name,first_name,state,abbreviation,name_state,current_tenure_len
0,Senate,"Slotkin, Elissa",2025,Slotkin,Elissa,Michigan,MI,SlotkinEMI,0.0
1,House of Representatives,"Grijalva, Raúl M.",2003,Grijalva,Raúl,Arizona,AZ,GrijalvaRAZ,22.0
2,Senate,"Banks, Jim",2025,Banks,Jim,Indiana,IN,BanksJIN,0.0
3,Senate,"Young, Todd",2017,Young,Todd,Indiana,IN,YoungTIN,8.0
4,Senate,"Wyden, Ron",1996,Wyden,Ron,Oregon,OR,WydenROR,29.0


# Final data preparation

## Merge current sponsor tenure length to bills

In [ ]:
# Create identifier field to match with current sponsor tenure DF
# last name, first initial, state abbreviation
bills_data_parsed_df['name_state'] = bills_data_parsed_df['sponsor_last_name'] + bills_data_parsed_df['sponsor_first_name'].str[0] + bills_data_parsed_df['sponsor_state']

In [ ]:
# replace some bill name_state identifiers with manual corrections to match tenure dataset

name_state_correct = {
    "Gonzalez-ColonJPR": None,
    "GonzalesETX": "GonzalesTTX",
    "GoodRVA": "GoodBVA",
    "Jackson-LeeSTX": "Jackson LeeSTX",
    "BarrGKY": "BarrAKY",
    "SanchezLCA": "SánchezLCA",
    "Gluesenkamp PerezMWA": "PerezMWA",
    "MooreFAL": "MooreBAL",
    "RochesterLDE": "Blunt RochesterLDE",
    "FernandezTNM": "Leger FernandezTNM",
    "CardenasTCA": "CárdenasTCA",
    "SherrillRNJ": "SherrillMNJ",
    "SteubeGFL": "SteubeWFL",
    "Van DuyneETX": "Van DuyneBTX",
    "VelazquezNNY": "VelázquezNNY",
    "HillFAR": "HillJAR",
    "BarraganNCA": "BarragánNCA",
    "FletcherETX": "FletcherLTX",
    "GarciaJIL": "GarcíaJIL",
    "CorreaLCA": "CorreaJCA",
    "Devolder-SantosGNY": "SantosGNY",
    "BishopJNC": "BishopDNC",
    "LaturnerJKS": "LaTurnerJKS",
    "FergusonDGA": "FergusonAGA"
}


In [ ]:
bills_data_parsed_df['name_state'] = bills_data_parsed_df['name_state'].replace(name_state_correct)

In [ ]:
# left merge (bills and member tenure)

bills_data_parsed_final = pd.merge(bills_data_parsed_df, member_data_parsed_df, on = 'name_state', how = 'left')

In [ ]:
bills_data_parsed_final.head()

,title,bill_number,body,bill_type,status,sponsor_party,sponsor_people_id,sponsor_hash,sponsor_first_name,sponsor_last_name,...,reported_to_floor,name_state,recent_chamber,name,start_year,last_name,first_name,state,abbreviation,current_tenure_len
0,Defense Production and Active Pharmaceutical I...,HB6895,H,B,1,D,20104,2guyleny,Elissa,Slotkin,...,False,SlotkinEMI,Senate,"Slotkin, Elissa",2025.0,Slotkin,Elissa,Michigan,MI,0.0
1,EEOC Transparency and Accountability Act,HB1054,H,B,1,R,11077,t1xxilhf,Tim,Walberg,...,False,WalbergTMI,House of Representatives,"Walberg, Tim",2011.0,Walberg,Tim,Michigan,MI,14.0
2,Natural Disaster Recovery Program Act of 2024,HB9750,H,B,1,R,16476,fds8xyci,David,Rouzer,...,True,RouzerDNC,House of Representatives,"Rouzer, David",2015.0,Rouzer,David,North Carolina,NC,10.0
3,Year-Round E15 Act of 2024,HB8052,H,B,1,R,24053,d6p4ilmt,Zachary,Nunn,...,False,NunnZIA,House of Representatives,"Nunn, Zachary",2023.0,Nunn,Zachary,Iowa,IA,2.0
4,Health Equity and MENA Community Inclusion Act...,HB2730,H,B,1,D,20115,pie8gfo6,Rashida,Tlaib,...,False,TlaibRMI,House of Representatives,"Tlaib, Rashida",2019.0,Tlaib,Rashida,Michigan,MI,6.0


In [ ]:
# find failed matches

failed_matches = bills_data_parsed_final[bills_data_parsed_final['current_tenure_len'].isna()]

In [ ]:
(list(failed_matches['name_state'].unique()))

[None]

## Calculate Congress progression metric for each bill

In [ ]:
# Convert introduction date to datetime object

bills_data_parsed_final['introduction_date'] = pd.to_datetime(bills_data_parsed_final['introduction_date'])

In [ ]:
bills_data_parsed_final['progression_metric'] = (bills_data_parsed_final['introduction_date'] -congress_start_date_obj)*100 / (congress_end_date_obj-congress_start_date_obj)

## Save final data without dummy columns

### As pickle object

In [ ]:
import pickle
from google.colab import drive

# connect to drive
drive.mount('/content/gdrive/')

# Save to file
with open('/content/gdrive/XXX', 'wb') as file:
    pickle.dump(bills_data_parsed_final, file)
    print('Done!')

Drive already mounted at /content/gdrive/; to attempt to forcibly remount, call drive.mount("/content/gdrive/", force_remount=True).
Done!


## Dummy code fields

In [ ]:
# list of columns in dataset
bills_data_parsed_final.columns

Index(['title', 'bill_number', 'body', 'bill_type', 'status', 'sponsor_party',
       'sponsor_people_id', 'sponsor_hash', 'sponsor_first_name',
       'sponsor_last_name', 'sponsor_state', 'num_cosponsors',
       'cosponsor_parties', 'cosponsor_states', 'num_cosponsor_states',
       'per_cosponsor_parties_D', 'per_cosponsor_parties_R',
       'committees_house', 'sub-committees_house', 'num_committees_house',
       'num_subcommittees_house', 'subject_areas', 'num_subject_areas',
       'num_related_bills', 'introduction_date', 'text_file_size',
       'bill_type_mapped', 'bill_number_clean', 'bill_id_concat',
       'reported_to_floor', 'name_state', 'recent_chamber', 'name',
       'start_year', 'last_name', 'first_name', 'state', 'abbreviation',
       'current_tenure_len', 'progression_metric'],
      dtype='object')

In [ ]:
df = bills_data_parsed_final.copy()

# Process regular categorical columns
regular_cols = ['bill_type', 'sponsor_party']
regular_dummies = []
for col in regular_cols:
    if col in df.columns:
        dummies = pd.get_dummies(df[col], prefix=col)
        regular_dummies.append(dummies)

# Process list columns - ensuring each unique value gets its own column
list_cols = ['committees_house', 'sub-committees_house', 'subject_areas']
list_dummies = []

for col in list_cols:
    if col in df.columns:
        # First, make sure we're working with lists
        if not all(isinstance(x, list) for x in df[col].dropna()):
            # Convert string representations of lists to actual lists if needed
            df[col] = df[col].apply(lambda x: eval(x) if isinstance(x, str) and x.startswith('[') else x)

        # Find all unique values across all lists
        unique_values = set()
        for item_list in df[col].dropna():
            if isinstance(item_list, list):
                unique_values.update(item_list)

        # Create a dictionary to hold all dummy columns for this list column
        dummy_dict = {}

        # Create a dummy column for each unique value
        for value in unique_values:
            # Create a clean column name
            clean_value = str(value).replace(' ', '_').replace('-', '_').replace('/', '_')
            col_name = f"{col}_{clean_value}"

            # Create the dummy column (True if value in list, False otherwise)
            dummy_dict[col_name] = df[col].apply(
                lambda x: True if isinstance(x, list) and value in x else False
            )

        # Create a DataFrame from the dictionary of dummy columns
        list_dummies.append(pd.DataFrame(dummy_dict, index=df.index))

# Combine all components using pd.concat
all_parts = [df] + regular_dummies + list_dummies
bills_data_parsed_final_dummies = pd.concat(all_parts, axis=1)


In [ ]:
bills_data_parsed_final_dummies.iloc[1]

,1
title,EEOC Transparency and Accountability Act
bill_number,HB1054
body,H
bill_type,B
status,1
...,...
subject_areas_Native_Americans,False
subject_areas_Sports_and_recreation,False
subject_areas_Government_operations_and_politics,False
subject_areas_Transportation_and_public_works,False


## Save final data

### As pickle object

In [ ]:
import pickle
from google.colab import drive

# connect to drive
drive.mount('/content/gdrive/')

# Save to file
with open('/content/gdrive/xxx', 'wb') as file:
    pickle.dump(bills_data_parsed_final_dummies, file)
    print('Done!')

Drive already mounted at /content/gdrive/; to attempt to forcibly remount, call drive.mount("/content/gdrive/", force_remount=True).
Done!
